In [1]:
import pandas as pd

# Reading in the Data

In [2]:
df = pd.read_csv("../datasets/raw/lbnl_wind_data.csv")
df.head()

,PPA Execution Date,Capacity (MW),Term (years),Region,Macro Region,Real 2024$/MWh
0,9/6/96,107.25,30.0,MISO,Central,42
1,11/27/96,25.08,30.0,West (non-ISO),West,80
2,3/11/97,108.75,20.0,MISO,Central,63
3,5/14/97,9.00,33.0,MISO,Central,64
4,7/16/97,70.50,19.0,MISO,Central,74


The columns are as follows:
- Date: in the formate MM/DD/YY, and not stored as a datetime type yet
- Capacity in Megawatts,
- Term in Years
- Region: The Mapping of states to region is provided in the source documentation, and I'll discuss it later below.
- Macro-Region
- Real 2024$/MWh

In [3]:
df.dtypes

PPA Execution Date     object
Capacity (MW)         float64
Term (years)          float64
Region                 object
Macro Region           object
Real 2024$/MWh          int64
dtype: object

# Cleaning and Saving Data

We need to group by year, so we create a new "Year" column.

In [4]:
df["Year"] = pd.to_datetime(df["PPA Execution Date"], format="%m/%d/%y").dt.year
df.head()

,PPA Execution Date,Capacity (MW),Term (years),Region,Macro Region,Real 2024$/MWh,Year
0,9/6/96,107.25,30.0,MISO,Central,42,1996
1,11/27/96,25.08,30.0,West (non-ISO),West,80,1996
2,3/11/97,108.75,20.0,MISO,Central,63,1997
3,5/14/97,9.00,33.0,MISO,Central,64,1997
4,7/16/97,70.50,19.0,MISO,Central,74,1997


All we are interested in is the cost per region per year. We can thus drop the Macro Region, Term, Capacity, and Date columns

In [5]:
df = df.drop(["PPA Execution Date", "Macro Region", "Term (years)", "Capacity (MW)"], axis=1)
df.head()

,Region,Real 2024$/MWh,Year
0,MISO,42,1996
1,West (non-ISO),80,1996
2,MISO,63,1997
3,MISO,64,1997
4,MISO,74,1997


Now, to get the average price per year per region, which we will be using later on, we need to group by region and year and take the mean of the prices.

We also are only interested in the years 2015 and later

In [6]:
df = df.groupby(["Region", "Year"])["Real 2024$/MWh"].mean().reset_index()
df = df[df["Year"] >= 2015]
df.head()

,Region,Year,Real 2024$/MWh
12,CAISO,2015,50.000000
13,CAISO,2016,49.666667
14,CAISO,2017,43.000000
15,CAISO,2018,42.333333
16,CAISO,2019,39.500000
